# Random Forest A/B · 17/11/2026 · Huy

Đo thay đổi MAE khi thêm thời tiết vào đặc trưng thời gian và ngày lễ. A và B dùng cùng tập train, validation và bộ tham số Random Forest từ lần đối chiếu ngày 16/11. Thời tiết cùng giờ được giả định có sẵn lúc dự đoán.

Dữ liệu gốc → Gộp theo giờ → Chia theo thời gian → Tạo đặc trưng A/B → Kiểm tra timestamp và mục tiêu → Fit trên train → Chấm validation.

Tập test được giữ cho đánh giá cuối. Chạy toàn bộ notebook để cập nhật CSV, JSON và biểu đồ trong thư mục thí nghiệm.

In [ ]:
from hashlib import file_digest
import json
from pathlib import Path
import sys

start = Path.cwd().resolve()
project = next(
    path for path in (start, *start.parents)
    if (path / "src" / "data.py").is_file()
)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor

from src.data import DATA_PATH, fill_missing_weather, load_hourly_splits
from src.evaluation import regression_metrics, weekday_hour_baseline
from src.features import TIME_FEATURES, WEATHER_FEATURES, add_time_features, add_weather_features

experiment = project / "experiments" / "17-11-2026_Huy_Xulydulieu"
data_path = DATA_PATH if DATA_PATH.is_file() else project / "files" / "Metro_Interstate_Traffic_Volume.csv.gz"

features_a = list(TIME_FEATURES)
features_b = features_a + list(WEATHER_FEATURES)
feature_sets = {"A": features_a, "B": features_b}
model_params = {
    "n_estimators": 400,
    "max_depth": 12,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "random_state": 42,
    "n_jobs": -1,
}
display(pd.Series(feature_sets, name="Đặc trưng"))

In [ ]:
splits = load_hourly_splits(data_path, mask_stuck_temp=False, complete_holidays=False)
development = {name: splits[name] for name in ("train", "validation")}
prepared = fill_missing_weather(development)
frames = {
    name: add_weather_features(add_time_features(part)).set_index(
        "date_time", verify_integrity=True
    )
    for name, part in prepared.items()
}
train, validation = frames["train"], frames["validation"]
X_validation = {
    config: validation.loc[:, columns].copy()
    for config, columns in feature_sets.items()
}
timeline = pd.DatetimeIndex(development["validation"]["date_time"], name="date_time")
if not timeline.is_unique or not timeline.is_monotonic_increasing or timeline.hasnans:
    raise ValueError("Timestamp validation phải duy nhất, có thứ tự và không bị thiếu.")
pd.testing.assert_index_equal(X_validation["A"].index, X_validation["B"].index, exact=True)
pd.testing.assert_index_equal(X_validation["A"].index, timeline, exact=True)
pd.testing.assert_series_equal(
    validation["traffic_volume"],
    development["validation"].set_index("date_time")["traffic_volume"],
)
pd.testing.assert_frame_equal(X_validation["A"], X_validation["B"][features_a])
if not train.index.max() < timeline.min():
    raise ValueError("Train phải kết thúc trước validation.")
if any(frame[features_b].isna().any().any() for frame in frames.values()):
    raise ValueError("Còn giá trị thiếu trong đặc trưng train hoặc validation.")

split_summary = {
    name: {
        "n_hours": len(part),
        "first_timestamp": str(part["date_time"].min()),
        "last_timestamp": str(part["date_time"].max()),
    }
    for name, part in splits.items()
}
print(f"A/B trùng tuyệt đối {len(timeline):,} timestamp validation trước khi fit.")
display(pd.DataFrame.from_dict(split_summary, orient="index"))

In [ ]:
predictions = pd.DataFrame({
    "traffic_volume": validation["traffic_volume"],
    "is_rush_hour": validation["is_rush_hour"].eq(1),
    "has_rain": validation["rain_1h"].gt(0),
    "baseline": weekday_hour_baseline(train, validation),
})
for config, columns in feature_sets.items():
    model = RandomForestRegressor(**model_params)
    model.fit(train[columns], train["traffic_volume"])
    predictions[f"rf_{config.lower()}"] = model.predict(X_validation[config])

prediction_columns = {"Baseline thứ–giờ": "baseline", "Random Forest A": "rf_a", "Random Forest B": "rf_b"}
actual = predictions["traffic_volume"]
metrics = pd.DataFrame.from_dict(
    {label: regression_metrics(actual, predictions[column]) for label, column in prediction_columns.items()},
    orient="index",
).rename_axis("model")
metrics.insert(0, "n_validation", len(predictions))
mae_gain = metrics.loc["Random Forest A", "MAE"] - metrics.loc["Random Forest B", "MAE"]
display(metrics.round(4))
print(f"MAE(A) − MAE(B): {mae_gain:+.4f} xe/giờ")

In [ ]:
groups = {
    "Giờ cao điểm": predictions["is_rush_hour"],
    "Giờ thường": ~predictions["is_rush_hour"],
    "Có mưa": predictions["has_rain"],
    "Không mưa": ~predictions["has_rain"],
}
errors_a = (actual - predictions["rf_a"]).abs()
errors_b = (actual - predictions["rf_b"]).abs()
slices = pd.DataFrame.from_records([
    {
        "group": name,
        "n_hours": int(mask.sum()),
        "MAE_A": errors_a[mask].mean(),
        "MAE_B": errors_b[mask].mean(),
        "MAE_A_minus_B": (errors_a - errors_b)[mask].mean(),
    }
    for name, mask in groups.items()
])
display(slices.round(4))

In [ ]:
outputs = experiment / "outputs"
figures = experiment / "figures"
outputs.mkdir(exist_ok=True)
figures.mkdir(exist_ok=True)
metrics.to_csv(outputs / "validation_metrics.csv")
predictions.to_csv(outputs / "validation_predictions.csv")
slices.to_csv(outputs / "validation_slices.csv", index=False)
with data_path.open("rb") as stream:
    data_hash = file_digest(stream, "sha256").hexdigest()
run_info = {
    "dataset": data_path.relative_to(project).as_posix(),
    "dataset_sha256": data_hash,
    "splits": split_summary,
    "validation_timestamps_identical": True,
    "feature_sets": feature_sets,
    "random_forest_params": model_params,
    "data_params": {"mask_stuck_temp": False, "complete_holidays": False},
    "mae_a_minus_b": float(mae_gain),
    "weather_assumption": "Same-hour weather is available at prediction time.",
    "rain_group": "rain_1h > 0 after shared preprocessing",
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
}
(outputs / "run_config.json").write_text(
    json.dumps(run_info, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
bars = ax.barh(metrics.index, metrics["MAE"], color=["#7A8793", "#426A92", "#268377"])
ax.invert_yaxis()
ax.bar_label(bars, fmt="%.2f", padding=5)
ax.set_xlim(0, metrics["MAE"].max() * 1.20)
ax.set_xlabel("MAE (xe/giờ)")
ax.set_title(f"Sai số trên cùng {len(predictions):,} giờ validation")
fig.savefig(figures / "validation_mae.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Nhận xét

- Trên cùng 6.086 giờ validation, thêm thời tiết giảm MAE từ **241,99** xuống **240,15 xe/giờ** (−0,76%). Mức giảm trong 1.091 giờ cao điểm là **6,56 xe/giờ**.
- RMSE tăng từ **380,93** lên **395,44 xe/giờ**, cho thấy B kém hơn theo thước đo nhạy với sai số lớn.
- Không có giờ nào có lượng mưa dương theo cột `rain_1h` sau tiền xử lý, nên chưa đánh giá được nhóm giờ mưa theo tiêu chí này.